In [ ]:


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve
from imblearn.over_sampling import SMOTE
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
import shap
import warnings
warnings.filterwarnings('ignore')

In [ ]:
class AdvancedDataPreprocessor:
    """Enhanced data preprocessing with augmentation"""

    def __init__(self):
        self.scaler = StandardScaler()
        self.label_encoder = LabelEncoder()

    def load_coimbra_dataset(self):
        """Load Breast Cancer Coimbra Dataset"""
        url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00451/dataR2.csv"
        try:
            df = pd.read_csv(url)
            print(f"Loaded Coimbra dataset: {df.shape}")
            return df
        except:
            print("Creating synthetic dataset...")
            return self._create_synthetic_data()

    def _create_synthetic_data(self, n_samples=500):
        """Create realistic synthetic dataset"""
        np.random.seed(42)

        # Generate correlated features
        n_healthy = int(n_samples * 0.55)
        n_cancer = n_samples - n_healthy

        # Healthy patients (lower risk biomarkers)
        healthy_data = {
            'Age': np.random.normal(45, 12, n_healthy),
            'BMI': np.random.normal(24, 3, n_healthy),
            'Glucose': np.random.normal(95, 10, n_healthy),
            'Insulin': np.random.gamma(2, 4, n_healthy),
            'HOMA': np.random.gamma(1.5, 1, n_healthy),
            'Leptin': np.random.gamma(2, 8, n_healthy),
            'Adiponectin': np.random.gamma(3, 3, n_healthy),
            'Resistin': np.random.gamma(2, 4, n_healthy),
            'MCP.1': np.random.gamma(2, 80, n_healthy),
            'Classification': np.ones(n_healthy)
        }

        # Cancer patients (higher risk biomarkers)
        cancer_data = {
            'Age': np.random.normal(55, 10, n_cancer),
            'BMI': np.random.normal(28, 4, n_cancer),
            'Glucose': np.random.normal(110, 15, n_cancer),
            'Insulin': np.random.gamma(3, 6, n_cancer),
            'HOMA': np.random.gamma(2.5, 1.5, n_cancer),
            'Leptin': np.random.gamma(3, 12, n_cancer),
            'Adiponectin': np.random.gamma(2, 2, n_cancer),
            'Resistin': np.random.gamma(3, 6, n_cancer),
            'MCP.1': np.random.gamma(3, 120, n_cancer),
            'Classification': np.ones(n_cancer) * 2
        }

        df_healthy = pd.DataFrame(healthy_data)
        df_cancer = pd.DataFrame(cancer_data)
        df = pd.concat([df_healthy, df_cancer], ignore_index=True)

        return df

    def create_multimodal_dataset(self, df, include_lifestyle=True):
        """Add lifestyle and demographic features"""

        if 'Classification' in df.columns:
            df['target'] = df['Classification'].apply(lambda x: 1 if x == 2 else 0)
            df = df.drop('Classification', axis=1)

        if include_lifestyle:
            np.random.seed(42)
            n = len(df)

            # Smoking (correlated with age and cancer risk)
            df['smoking_status'] = np.random.choice([0, 1, 2], n, p=[0.5, 0.3, 0.2])
            df.loc[df['target'] == 1, 'smoking_status'] = np.random.choice([0, 1, 2], sum(df['target'] == 1), p=[0.3, 0.4, 0.3])

            # Alcohol consumption
            df['alcohol_units'] = np.random.gamma(2, 2, n)

            # Exercise (inversely correlated with BMI)
            df['exercise_hours'] = np.maximum(0, 7 - 0.15 * df['BMI'] + np.random.normal(0, 1.5, n))

            # Diet quality
            df['diet_quality'] = np.random.uniform(3, 9, n)

            # Family history (higher in cancer group)
            df['family_history'] = 0
            df.loc[df['target'] == 1, 'family_history'] = np.random.choice([0, 1], sum(df['target'] == 1), p=[0.65, 0.35])
            df.loc[df['target'] == 0, 'family_history'] = np.random.choice([0, 1], sum(df['target'] == 0), p=[0.85, 0.15])

        return df

    def preprocess_data(self, df, target_col='target'):
        """Enhanced preprocessing with feature engineering"""

        X = df.drop(target_col, axis=1)
        y = df[target_col]

        # Feature engineering: create interaction features
        if 'Glucose' in X.columns and 'Insulin' in X.columns:
            X['glucose_insulin_ratio'] = X['Glucose'] / (X['Insulin'] + 1)

        if 'BMI' in X.columns and 'Age' in X.columns:
            X['bmi_age_interaction'] = X['BMI'] * X['Age'] / 100

        # Handle missing values
        X = X.fillna(X.median())

        feature_names = X.columns.tolist()

        # Normalize features
        X_scaled = self.scaler.fit_transform(X)
        X_scaled = pd.DataFrame(X_scaled, columns=feature_names)

        return X_scaled, y, feature_names

In [ ]:
class DeepNeuralNetworkArchitectures:
    """Multiple deep learning architectures"""

    @staticmethod
    def create_dense_network(input_dim, architecture='deep'):
        """
        Create dense neural network
        Architectures: 'shallow', 'medium', 'deep', 'very_deep'
        """

        models_config = {
            'shallow': [64, 32],
            'medium': [128, 64, 32],
            'deep': [256, 128, 64, 32],
            'very_deep': [512, 256, 128, 64, 32]
        }

        layers_config = models_config.get(architecture, models_config['deep'])

        model = keras.Sequential(name=f'DenseNet_{architecture}')

        # Input layer
        model.add(layers.Input(shape=(input_dim,)))

        # Hidden layers
        for i, units in enumerate(layers_config):
            model.add(layers.Dense(
                units,
                activation='relu',
                kernel_regularizer=keras.regularizers.l2(0.001),
                kernel_initializer='he_normal',
                name=f'dense_{i+1}'
            ))
            model.add(layers.BatchNormalization(name=f'bn_{i+1}'))

            # Adaptive dropout (higher for earlier layers)
            dropout_rate = 0.5 - (i * 0.1)
            dropout_rate = max(dropout_rate, 0.2)
            model.add(layers.Dropout(dropout_rate, name=f'dropout_{i+1}'))

        # Output layer
        model.add(layers.Dense(1, activation='sigmoid', name='output'))

        return model

    @staticmethod
    def create_residual_network(input_dim):
        """Create deep network with residual connections"""

        inputs = layers.Input(shape=(input_dim,), name='input')

        # First block
        x = layers.Dense(256, activation='relu')(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.4)(x)

        # Residual block 1
        residual = x
        x = layers.Dense(256, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.3)(x)
        x = layers.Add()([x, residual])  # Skip connection

        # Second block
        x = layers.Dense(128, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.3)(x)

        # Residual block 2
        residual = layers.Dense(128)(x)
        x = layers.Dense(128, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.2)(x)
        x = layers.Add()([x, residual])

        # Final layers
        x = layers.Dense(64, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.2)(x)

        outputs = layers.Dense(1, activation='sigmoid', name='output')(x)

        model = models.Model(inputs=inputs, outputs=outputs, name='ResidualNet')

        return model

    @staticmethod
    def create_attention_network(input_dim):
        """Create network with attention mechanism"""

        inputs = layers.Input(shape=(input_dim,), name='input')

        # Feature extraction
        x = layers.Dense(256, activation='relu')(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.4)(x)

        # Attention mechanism
        attention = layers.Dense(256, activation='softmax', name='attention_weights')(x)
        x = layers.Multiply()([x, attention])

        # Deep processing
        x = layers.Dense(128, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.3)(x)

        x = layers.Dense(64, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.2)(x)

        outputs = layers.Dense(1, activation='sigmoid', name='output')(x)

        model = models.Model(inputs=inputs, outputs=outputs, name='AttentionNet')

        return model

In [ ]:
class AdvancedModelTrainer:
    """Advanced training with multiple techniques"""

    def __init__(self, model, model_name='model'):
        self.model = model
        self.model_name = model_name
        self.history = None

    def compile_model(self, learning_rate=0.001):
        """Compile model with advanced optimizer"""

        optimizer = keras.optimizers.Adam(
            learning_rate=learning_rate,
            beta_1=0.9,
            beta_2=0.999,
            epsilon=1e-07,
            clipnorm=1.0  # Gradient clipping
        )

        self.model.compile(
            optimizer=optimizer,
            loss='binary_crossentropy',
            metrics=[
                'accuracy',
                keras.metrics.AUC(name='auc'),
                keras.metrics.Precision(name='precision'),
                keras.metrics.Recall(name='recall'),
                keras.metrics.TruePositives(name='tp'),
                keras.metrics.FalseNegatives(name='fn')
            ]
        )

    def get_callbacks(self):
        """Create advanced callbacks"""

        callback_list = [
            # Early stopping with patience
            callbacks.EarlyStopping(
                monitor='val_auc',
                patience=20,
                restore_best_weights=True,
                mode='max',
                verbose=1
            ),

            # Reduce learning rate on plateau
            callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=10,
                min_lr=1e-7,
                verbose=1
            ),

            # Model checkpoint
            callbacks.ModelCheckpoint(
                f'best_{self.model_name}.keras',
                monitor='val_auc',
                mode='max',
                save_best_only=True,
                verbose=1
            ),

            # Learning rate scheduler
            callbacks.LearningRateScheduler(
                lambda epoch: 0.001 * 0.95 ** epoch
            )
        ]

        return callback_list

    def train(self, X_train, y_train, X_val, y_val, epochs=100, batch_size=32):
        """Train model with advanced techniques"""

        self.history = self.model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=self.get_callbacks(),
            class_weight=self._calculate_class_weights(y_train),
            verbose=1
        )

        return self.history

    def _calculate_class_weights(self, y):
        """Calculate class weights for imbalanced data"""
        from sklearn.utils.class_weight import compute_class_weight

        classes = np.unique(y)
        weights = compute_class_weight('balanced', classes=classes, y=y)

        return dict(zip(classes, weights))

In [ ]:
class EnsemblePredictor:
    """Ensemble multiple models for better predictions"""

    def __init__(self):
        self.models = []
        self.weights = []

    def add_model(self, model, weight=1.0):
        """Add model to ensemble"""
        self.models.append(model)
        self.weights.append(weight)

    def predict(self, X):
        """Ensemble prediction with weighted voting"""
        predictions = []

        for model, weight in zip(self.models, self.weights):
            pred = model.predict(X, verbose=0)
            predictions.append(pred * weight)

        # Weighted average
        ensemble_pred = np.average(predictions, axis=0)

        return ensemble_pred

In [ ]:
def main():

    preprocessor = AdvancedDataPreprocessor()

    # Load dataset
    df = preprocessor.load_coimbra_dataset()
    df = preprocessor.create_multimodal_dataset(df, include_lifestyle=True)

    # Preprocess
    X, y, feature_names = preprocessor.preprocess_data(df)
    print(f"Dataset shape: {X.shape}")
    print(f"Features: {len(feature_names)}")
    print(f"Class distribution: {dict(y.value_counts())}")

In [ ]:
def main():

    preprocessor = AdvancedDataPreprocessor()

    # Load dataset
    df = preprocessor.load_coimbra_dataset()
    df = preprocessor.create_multimodal_dataset(df, include_lifestyle=True)

    # Preprocess
    X, y, feature_names = preprocessor.preprocess_data(df)
    print(f"Dataset shape: {X.shape}")
    print(f"Features: {len(feature_names)}")
    print(f"Class distribution: {dict(y.value_counts())}")

    # Split data
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

    # Balance training data
    smote = SMOTE(random_state=42)
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)
    print(f"Training data balanced: {len(X_train_balanced)} samples")

    architectures = DeepNeuralNetworkArchitectures()

    # Create multiple models
    models_to_train = {
        'deep_dense': architectures.create_dense_network(X.shape[1], 'deep'),
        'very_deep_dense': architectures.create_dense_network(X.shape[1], 'very_deep'),
        'residual': architectures.create_residual_network(X.shape[1]),
        'attention': architectures.create_attention_network(X.shape[1])
    }

    trained_models = {}
    results = {}

    for name, model in models_to_train.items():
        print(f"\n--- Training {name} ---")

        trainer = AdvancedModelTrainer(model, name)
        trainer.compile_model(learning_rate=0.001)

        print(f"Model: {name}")
        print(f"Parameters: {model.count_params():,}")

        # Train
        history = trainer.train(
            X_train_balanced, y_train_balanced,
            X_val, y_val,
            epochs=100,
            batch_size=16
        )

        trained_models[name] = model


    for name, model in trained_models.items():
        print(f"\n--- Evaluating {name} ---")

        # Predictions
        y_pred_proba = model.predict(X_test, verbose=0)
        y_pred = (y_pred_proba > 0.5).astype(int)

        # Metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_pred_proba)

        cm = confusion_matrix(y_test, y_pred)
        tn, fp, fn, tp = cm.ravel()
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

        results[name] = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'auc': auc,
            'fnr': fnr
        }

        print(f"Accuracy:  {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall:    {recall:.4f}")
        print(f"F1-Score:  {f1:.4f}")
        print(f"AUC:       {auc:.4f}")
        print(f"FNR:       {fnr:.4f} (CRITICAL)")


    ensemble = EnsemblePredictor()

    # Add models with weights based on performance
    for name, model in trained_models.items():
        weight = results[name]['auc']  # Use AUC as weight
        ensemble.add_model(model, weight)

    # Ensemble prediction
    y_ensemble_proba = ensemble.predict(X_test)
    y_ensemble_pred = (y_ensemble_proba > 0.5).astype(int).flatten()

    # Ensemble metrics
    ensemble_accuracy = accuracy_score(y_test, y_ensemble_pred)
    ensemble_auc = roc_auc_score(y_test, y_ensemble_proba)
    ensemble_recall = recall_score(y_test, y_ensemble_pred)

    cm_ensemble = confusion_matrix(y_test, y_ensemble_pred)
    tn, fp, fn, tp = cm_ensemble.ravel()
    ensemble_fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

    print("\n" + "="*80)
    print("ENSEMBLE MODEL RESULTS")
    print("="*80)
    print(f"Accuracy:  {ensemble_accuracy:.4f}")
    print(f"AUC:       {ensemble_auc:.4f}")
    print(f"Recall:    {ensemble_recall:.4f}")
    print(f"FNR:       {ensemble_fnr:.4f} (CRITICAL: Lower is better)")
    print("="*80)

    comparison_df = pd.DataFrame(results).T
    print("\nModel Performance Comparison:")
    print(comparison_df.round(4))

    # Find best model
    best_model_name = comparison_df['auc'].idxmax()
    print(f"\n Best Model: {best_model_name}")
    print(f"   AUC: {comparison_df.loc[best_model_name, 'auc']:.4f}")
    print(f"   FNR: {comparison_df.loc[best_model_name, 'fnr']:.4f}")

    # Save best model
    best_model = trained_models[best_model_name]
    best_model.save('best_breast_cancer_model.keras')
    print(f"Best model saved: best_breast_cancer_model.keras")

    # Save scaler
    import joblib
    joblib.dump(preprocessor.scaler, 'scaler.pkl')
    print("Scaler saved: scaler.pkl")

    # Save feature names
    with open('feature_names.json', 'w') as f:
        import json
        json.dump(feature_names, f)
    print("Feature names saved: feature_names.json")

    print("\n" + "="*80)
    print("DEEP LEARNING MODEL TRAINING COMPLETE!")
    print("="*80)
    print("\nFiles created:")
    print("  1. best_breast_cancer_model.keras - Best performing model")
    print("  2. scaler.pkl - Feature scaler")
    print("  3. feature_names.json - Feature list")
    print("  4. best_*.keras - Checkpoints for each model")
    print("\nYou can now use these files in your web application!")
    print("="*80)


if __name__ == "__main__":
    main()

Loaded Coimbra dataset: (116, 10)
Dataset shape: (116, 16)
Features: 16
Class distribution: {1: np.int64(64), 0: np.int64(52)}
Training data balanced: 90 samples

--- Training deep_dense ---
Model: deep_dense
Parameters: 49,537
Epoch 1/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4322 - auc: 0.4656 - fn: 15.8333 - loss: 1.9056 - precision: 0.3948 - recall: 0.3665 - tp: 10.6667
Epoch 1: val_auc improved from -inf to 0.61111, saving model to best_deep_dense.keras
6/6 ━━━━━━━━━━━━━━━━━━━━ 18s 2s/step - accuracy: 0.4355 - auc: 0.4669 - fn: 17.2857 - loss: 1.9043 - precision: 0.4030 - recall: 0.3745 - tp: 11.8571 - val_accuracy: 0.5882 - val_auc: 0.6111 - val_fn: 7.0000 - val_loss: 1.6949 - val_precision: 1.0000 - val_recall: 0.2222 - val_tp: 2.0000 - learning_rate: 0.0010
Epoch 2/100
1/6 ━━━━━━━━━━━━━━━━━━━━ 36s 7s/step - accuracy: 0.5000 - auc: 0.4297 - fn: 5.0000 - loss: 1.9103 - precision: 0.5000 - recall: 0.3750 - tp: 3.0000
Epoch 2: val_auc did not improve from 0.61111
6/6 ━━

Accuracy:  0.7778
Precision: 0.8000
Recall:    0.8000
F1-Score:  0.8000
AUC:       0.8625
FNR:       0.2000 (CRITICAL)

--- Evaluating very_deep_dense ---


Accuracy:  0.6111
Precision: 0.5882
Recall:    1.0000
F1-Score:  0.7407
AUC:       0.4625
FNR:       0.0000 (CRITICAL)

--- Evaluating residual ---
Accuracy:  0.5000
Precision: 0.5294
Recall:    0.9000
F1-Score:  0.6667
AUC:       0.5250
FNR:       0.1000 (CRITICAL)

--- Evaluating attention ---
Accuracy:  0.4444
Precision: 0.0000
Recall:    0.0000
F1-Score:  0.0000
AUC:       0.8000
FNR:       1.0000 (CRITICAL)

ENSEMBLE MODEL RESULTS
Accuracy:  0.4444
AUC:       0.8250
Recall:    0.0000
FNR:       1.0000 (CRITICAL: Lower is better)

Model Performance Comparison:
                 accuracy  precision  recall      f1     auc  fnr
deep_dense         0.7778     0.8000     0.8  0.8000  0.8625  0.2
very_deep_dense    0.6111     0.5882     1.0  0.7407  0.4625  0.0
residual           0.5000     0.5294     0.9  0.6667  0.5250  0.1
attention          0.4444     0.0000     0.0  0.0000  0.8000  1.0

 Best Model: deep_dense
   AUC: 0.8625
   FNR: 0.2000
Best model saved: best_breast_cancer_model.k